<a href="https://colab.research.google.com/github/moucheng2017/my_simsiam/blob/26-linear-eval-cifar10/notebooks/linear_eval/linear-eval-simclr-cifar10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SimCLR Linear Evaluation on CIFAR-10 (Google Colab)

This notebook runs linear evaluation on a trained SimCLR checkpoint saved on Google Drive using the CIFAR-10 dataset.

A fixed subset of `eval.source_pool_size` unique images is drawn from the CIFAR-10 training set and used to train the linear classifier on top of the frozen backbone. If `eval.source_pool_size` is not set (or `None`), the full CIFAR-10 training set is used. The linear classifier is then evaluated on the full CIFAR-10 test set.

If the GitHub repository is private, create a GitHub fine-grained personal access token with repository read access and paste it when prompted in the setup cell.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ModuleNotFoundError:
    print('google.colab is only available inside Google Colab; skipping Drive mount.')

In [ ]:
import getpass
import os
import subprocess
from pathlib import Path
from urllib.parse import quote

PUBLIC_REPO_URL = 'https://github.com/moucheng2017/my_simsiam.git'
BRANCH = 'main'
REPO_DIR = Path('/content/my_simsiam')
GITHUB_TOKEN = ""  # Set to a token string here if the repo is private.


def _redact_cmd_arg(arg):
    if isinstance(arg, str) and arg.startswith('https://oauth2:') and '@' in arg:
        return 'https://oauth2:***@' + arg.split('@', 1)[1]
    return arg


def run(cmd, cwd=None):
    print('+', ' '.join(_redact_cmd_arg(arg) for arg in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get('GITHUB_TOKEN')
except Exception:
    pass

repo_url = PUBLIC_REPO_URL
if GITHUB_TOKEN:
    repo_url = PUBLIC_REPO_URL.replace('https://', f'https://oauth2:{quote(GITHUB_TOKEN, safe="")}@')


def sync_repo():
    if GITHUB_TOKEN:
        run(['git', 'remote', 'set-url', 'origin', repo_url], cwd=REPO_DIR)
    else:
        print('No GitHub token found; keeping the existing origin URL for fetch/pull.')
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)

os.chdir('/content')

if not REPO_DIR.exists():
    try:
        run(['git', 'clone', '--branch', BRANCH, repo_url, str(REPO_DIR)])
    except subprocess.CalledProcessError as exc:
        if not GITHUB_TOKEN:
            print('\nClone failed without a token. If this repository is private, paste a GitHub token with read access.')
            token = getpass.getpass('GitHub token (leave blank to stop): ').strip()
            if token:
                repo_url = PUBLIC_REPO_URL.replace('https://', f'https://oauth2:{quote(token, safe="")}@')
                run(['git', 'clone', '--branch', BRANCH, repo_url, str(REPO_DIR)])
            else:
                raise RuntimeError('Repository clone was skipped because no token was provided.') from exc
        else:
            raise
else:
    print(f'Repo already exists at {REPO_DIR}; pulling latest changes from {BRANCH}.')
    try:
        sync_repo()
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Failed to fetch the latest repo changes. If the repository is private, make sure a GITHUB_TOKEN Colab secret is set or rerun after deleting /content/my_simsiam so the clone step can prompt for a token.'
        ) from exc

if not REPO_DIR.exists():
    raise FileNotFoundError(f'Expected repository at {REPO_DIR}, but it was not created.')

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
run(['pip', 'install', '-r', 'requirements_colab.txt'], cwd=REPO_DIR)

In [ ]:
# CIFAR-10 is downloaded automatically by torchvision the first time this cell runs.
# The dataset is ~170 MB and will be saved to Google Drive so it only downloads once.
import os
import torchvision

DATA_DIR = '/content/drive/MyDrive/SSL_exps/data'
os.makedirs(DATA_DIR, exist_ok=True)

print('Checking / downloading CIFAR-10 ...')
torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)
print('CIFAR-10 is ready.')

In [ ]:
import copy
import json
import datetime
import yaml


def save_eval_results(eval_result, run_name, checkpoint_path, config_file, overrides):
    log_dir = os.path.join(eval_result['paths']['log_dir'], run_name)
    os.makedirs(log_dir, exist_ok=True)

    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    log_entry = {
        'timestamp': timestamp,
        'checkpoint': checkpoint_path,
        'accuracy': eval_result['accuracy'],
    }

    log_path = os.path.join(log_dir, f'linear_eval_{timestamp}.json')
    with open(log_path, 'w') as f:
        json.dump(log_entry, f, indent=2)

    print(f'Results saved to {log_path}')

    # Save the resolved config (base config + overrides) as YAML.
    with open(config_file, 'r') as f:
        resolved_config = yaml.safe_load(f)

    def _deep_update(base, updates):
        for k, v in updates.items():
            if isinstance(v, dict) and isinstance(base.get(k), dict):
                _deep_update(base[k], v)
            else:
                base[k] = v

    _deep_update(resolved_config, copy.deepcopy(overrides))

    config_save_path = os.path.join(log_dir, 'config.yaml')
    with open(config_save_path, 'w') as f:
        yaml.dump(resolved_config, f, default_flow_style=False, sort_keys=False)

    print(f'Config saved to {config_save_path}')


In [ ]:
from colab_utils import linear_eval_from_colab

# Path to the SimCLR checkpoint saved on Google Drive.
# Update this to match the checkpoint you want to evaluate.
CHECKPOINT_PATH = '/content/drive/MyDrive/SSL_exps/checkpoints/0524111121_meta_training_cifar10/episode_8/episode_8_0525083402.pth'
RUN_NAME = 'linear-eval-meta-pseudo-0524224426_random-sampling-bs2048-new-scheduler-iter20-episode50-warmup0.1-episode8'
CONFIG_FILE = 'configs/linear_evals/simclr_cifar_eval_sgd.yaml'
OVERRIDES = {
    'name': RUN_NAME,
    'eval': {
        'source_pool_size': None,  # Set to None to use the full training set.
        'source_subset_seed': 0,
    }
}

eval_result = linear_eval_from_colab(
    config_file=CONFIG_FILE,
    checkpoint_path=CHECKPOINT_PATH,
    project_name='SSL_exps',
    use_drive=True,
    device='cuda',
    download=False,  # CIFAR-10 is pre-downloaded by the cell above.
    overrides=OVERRIDES,
)
eval_result

save_eval_results(eval_result, RUN_NAME, CHECKPOINT_PATH, CONFIG_FILE, OVERRIDES)
